# PyDBAdminKit 0.3.0 — CLI Experiment Lab

Notebook de démonstration de la CLI `pydbadminkit` depuis Python/Jupyter.

- aucune valeur de secret n’est enregistrée ;
- les mutations réelles sont **désactivées par défaut** ;
- les opérations critiques sont montrées en dry-run uniquement ;
- ce notebook complète `00 - Setup.ipynb`, qui exerce directement l’API Python.


In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

from pydbadminkit.bootstrap import resolve_connection

In [3]:
import os

os.environ["PYDBADMIN_NATIVE_PASSWORD"] = "postgres"
print(os.environ.get("PYDBADMIN_NATIVE_PASSWORD"))

postgres


In [4]:
CONNECTION_PROFILE = "local-native"  # "local" pour Docker
RUN_MUTATIONS = False


def find_config_path() -> Path:
    for candidate in (Path.cwd() / "config.toml", Path.cwd().parent / "config.toml"):
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("config.toml introuvable")


CONFIG_PATH = find_config_path()
PASSWORD_ENV = (
    "PYDBADMIN_NATIVE_PASSWORD"
    if CONNECTION_PROFILE == "local-native"
    else "PYDBADMIN_LOCAL_PASSWORD"
)

if not os.environ.get(PASSWORD_ENV):
    raise RuntimeError(f"Définissez {PASSWORD_ENV} avant de lancer le notebook.")

resolved_config = resolve_connection(CONNECTION_PROFILE, CONFIG_PATH)
print("Profil       :", CONNECTION_PROFILE)
print("Config       :", CONFIG_PATH)
print("Environment  :", resolved_config.environment)
print("Mutations    :", RUN_MUTATIONS)

Profil       : local-native
Config       : C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml
Environment  : development
Mutations    : False


In [5]:
def run(args: list[str], *, expect_success: bool = True):
    base = [
        sys.executable,
        "-m",
        "pydbadminkit",
        "--connection",
        CONNECTION_PROFILE,
        "--config",
        str(CONFIG_PATH),
    ]
    command = base + args
    print("▶", " ".join(command[2:]))
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print("STDERR:", result.stderr.rstrip())
    if expect_success and result.returncode != 0:
        raise RuntimeError(f"Commande échouée avec le code {result.returncode}")
    return result


def run_json(args: list[str]):
    result = run(["--output", "json", *args])
    return json.loads(result.stdout)


def run_mutation(args: list[str]):
    if not RUN_MUTATIONS:
        print("[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)")
        return None
    if resolved_config.read_only or str(resolved_config.environment) not in {
        "development",
        "testing",
    }:
        raise RuntimeError(
            "Le notebook n'autorise les mutations qu'en development/testing, read_only=false"
        )
    return run(args)

## 1. Foundation


In [6]:
run(["--version"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml --version
pydbadminkit 0.5.0b1


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', '--version'], returncode=0, stdout='pydbadminkit 0.5.0b1\n', stderr='')

In [7]:
run(["connection", "test"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml connection test
Connection OK
Engine: postgresql
Version: 17.10
Database: pydbadmin_dev
User: postgres
Latency: 236.04 ms


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', 'connection', 'test'], returncode=0, stdout='Connection OK\nEngine: postgresql\nVersion: 17.10\nDatabase: pydbadmin_dev\nUser: postgres\nLatency: 236.04 ms\n', stderr='')

In [8]:
run(["capability", "list"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml capability list
NAME	STATUS	REASON
backup.create	unavailable_tool	Required tool 'pg_dump' was not found.
backup.restore	unavailable_tool	Required restore tool(s) not found: pg_restore, psql.
backup.validate	available	-
catalog.database.describe	available	-
catalog.database.list	available	-
catalog.index.describe	available	-
catalog.index.list	available	-
catalog.schema.describe	available	-
catalog.schema.list	available	-
catalog.table.describe	available	-
catalog.table.list	available	-
catalog.view.describe	available	-
catalog.view.list	available	-
connection.test	available	-
maintenance.analyze	available	-
maintenance.reindex	available	-
maintenance.vacuum	available	-
postgres.reindex.concurrently	available	-
postgres.reindex.progress	available	-
postgres.vacuum.progress	available	-
runtime.blocking.list	available	-
runtime.lock.list	available	-
runtime.query.cancel	availab

CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', 'capability', 'list'], returncode=0, stdout="NAME\tSTATUS\tREASON\nbackup.create\tunavailable_tool\tRequired tool 'pg_dump' was not found.\nbackup.restore\tunavailable_tool\tRequired restore tool(s) not found: pg_restore, psql.\nbackup.validate\tavailable\t-\ncatalog.database.describe\tavailable\t-\ncatalog.database.list\tavailable\t-\ncatalog.index.describe\tavailable\t-\ncatalog.index.list\tavailable\t-\ncatalog.schema.describe\tavailable\t-\ncatalog.schema.list\tavailable\t-\ncatalog.table.describe\tavailable\t-\ncatalog.table.list\tavailable\t-\ncatalog.view.describe\tavailable\t-\ncatalog.view.list\tavailable\t-\nconnection.test\tavailable\t-\nmaintenance.analyze\tavailable\t-\nmaintenance.reindex\tavailable\t-\nmaintenance.vacuum\tavailable\

In [ ]:
run(["server", "info"])

## 2. Object Explorer


In [9]:
run(["database", "list"])
run(["schema", "list"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml database list
NAME	OWNER	ENCODING	CONNECTIONS	SIZE_BYTES
postgres	postgres	UTF8	yes	8033971
pydbadmin_dev	postgres	UTF8	yes	8050355
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml schema list
NAME	OWNER	SYSTEM
public	pg_database_owner	no


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', 'schema', 'list'], returncode=0, stdout='NAME\tOWNER\tSYSTEM\npublic\tpg_database_owner\tno\n', stderr='')

In [10]:
databases = run_json(["database", "list"])
print("Type:", type(databases).__name__)
print("Nombre:", len(databases))

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml --output json database list
[
  {
    "name": "postgres",
    "owner": "postgres",
    "encoding": "UTF8",
    "collation": "French_France.1252",
    "allow_connections": true,
    "connection_limit": -1,
    "size_bytes": 8033971
  },
  {
    "name": "pydbadmin_dev",
    "owner": "postgres",
    "encoding": "UTF8",
    "collation": "French_France.1252",
    "allow_connections": true,
    "connection_limit": -1,
    "size_bytes": 8050355
  }
]
Type: list
Nombre: 2


### Objets de démonstration optionnels

La création de table/vue/index ci-dessous utilise `psycopg` uniquement si `RUN_MUTATIONS=True`.


In [11]:
if not RUN_MUTATIONS:
    print("[SKIP] Création des objets SQL de démonstration")
else:
    import psycopg

    password = os.environ[PASSWORD_ENV]
    sql_demo = """
    CREATE TABLE IF NOT EXISTS public.customers (
        id BIGSERIAL PRIMARY KEY,
        email TEXT NOT NULL UNIQUE,
        name TEXT NOT NULL,
        created_at TIMESTAMPTZ NOT NULL DEFAULT now()
    );
    CREATE INDEX IF NOT EXISTS customers_name_idx ON public.customers(name);
    CREATE OR REPLACE VIEW public.active_customers AS
    SELECT id, email, name FROM public.customers;
    """
    with (
        psycopg.connect(
            host=resolved_config.host,
            port=resolved_config.port,
            dbname=resolved_config.database,
            user=resolved_config.username,
            password=password,
        ) as connection,
        connection.cursor() as cursor,
    ):
        cursor.execute(sql_demo)
    print("Objets de démonstration créés.")

[SKIP] Création des objets SQL de démonstration


In [12]:
run(["table", "list", "--schema", "public"])
run(["view", "list", "--schema", "public"])
run(["index", "list", "--schema", "public"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml table list --schema public
NAME	OWNER	KIND	EST_ROWS	SIZE_BYTES
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml view list --schema public
NAME	OWNER	KIND
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml index list --schema public
NAME	TABLE	METHOD	UNIQUE	PRIMARY	VALID	READY	SIZE_BYTES


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', 'index', 'list', '--schema', 'public'], returncode=0, stdout='NAME\tTABLE\tMETHOD\tUNIQUE\tPRIMARY\tVALID\tREADY\tSIZE_BYTES\n', stderr='')

## 3. Security — inspection


In [13]:
run(["role", "list"])
run(["role", "list", "--login-only"])
run(["role", "describe", "postgres"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml role list
NAME	LOGIN	SUPERUSER	CREATEDB	CREATEROLE	REPLICATION	BYPASSRLS	SYSTEM
postgres	yes	yes	yes	yes	yes	yes	no
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml role list --login-only
NAME	LOGIN	SUPERUSER	CREATEDB	CREATEROLE	REPLICATION	BYPASSRLS	SYSTEM
postgres	yes	yes	yes	yes	yes	yes	no
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml role describe postgres
Name: postgres
Login: yes
Superuser: yes
Create DB: yes
Create role: yes
Replication: yes
Inherit: yes
Bypass RLS: yes
Connection limit: -1
Valid until: -

MEMBER OF
ROLE	GRANTOR	ADMIN

MEMBERS
MEMBER	GRANTOR	ADMIN


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', 'role', 'describe', 'postgres'], returncode=0, stdout='Name: postgres\nLogin: yes\nSuperuser: yes\nCreate DB: yes\nCreate role: yes\nReplication: yes\nInherit: yes\nBypass RLS: yes\nConnection limit: -1\nValid until: -\n\nMEMBER OF\nROLE\tGRANTOR\tADMIN\n\nMEMBERS\nMEMBER\tGRANTOR\tADMIN\n', stderr='')

In [14]:
run(["access", "list", "--role", "postgres"])
run(["effective-access", "list", "--role", "postgres"])
run(["ownership", "list", "--owner", "postgres"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml access list --role postgres
PRINCIPAL	OBJECT	TYPE	ACCESS	ISSUER	DELEGABLE
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml effective-access list --role postgres
PRINCIPAL	OBJECT	TYPE	ACCESS	SOURCES
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml ownership list --owner postgres
OWNER	OBJECT	TYPE
postgres	postgres	database
postgres	pydbadmin_dev	database
postgres	template0	database
postgres	template1	database


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', 'ownership', 'list', '--owner', 'postgres'], returncode=0, stdout='OWNER\tOBJECT\tTYPE\npostgres\tpostgres\tdatabase\npostgres\tpydbadmin_dev\tdatabase\npostgres\ttemplate0\tdatabase\npostgres\ttemplate1\tdatabase\n', stderr='')

## 4. Mutations — dry-run d’abord


In [15]:
run(["--dry-run", "role", "create", "demo_user", "--login"])
critical_plan = run_json(["--dry-run", "role", "create", "demo_super", "--superuser"])
print(json.dumps(critical_plan, indent=2, ensure_ascii=False))

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml --dry-run role create demo_user --login
Operation: security.role.create
Target: demo_user
Environment: development
Risk: medium
Confirmation: simple
Correlation ID: 3039f473-bed4-49d3-bbba-85f3695f1b90

EFFECTS
- Create role 'demo_user'.
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml --output json --dry-run role create demo_super --superuser
{
  "operation": "security.role.create",
  "target": "demo_super",
  "environment": "development",
  "risk": "critical",
  "confirmation": "type_target",
  "effects": [
    "Create role 'demo_super'."
  ],
  "warnings": [
    "Elevated role attributes requested: SUPERUSER."
  ],
  "correlation_id": "8a9d691b-0ec2-4019-ade1-246538cb1873"
}
{
  "operation": "security.role.create",
  "target": "demo_super",
  "environment": "development",
  "risk": "critical",
  "confirmation

### Cycle réel optionnel

Activez `RUN_MUTATIONS=True` uniquement sur une base de développement. Aucun rôle `SUPERUSER` n’est créé par ce notebook.


In [16]:
run_mutation(["--yes", "role", "create", "demo_user", "--login"])
run_mutation(["--yes", "role", "create", "demo_reader"])

[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)
[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)


In [17]:
run_mutation(["--yes", "role", "membership-add", "demo_reader", "demo_user"])
run_mutation(
    [
        "--yes",
        "access",
        "grant",
        "--role",
        "demo_user",
        "--object",
        "public.customers",
        "--access",
        "SELECT",
    ]
)

[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)
[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)


## 5. Sorties machine


In [18]:
server_json = run_json(["server", "info"])
print(json.dumps(server_json, indent=2, ensure_ascii=False))
run(["--output", "yaml", "database", "list"])

▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml --output json server info
{
  "engine": "postgresql",
  "version": {
    "major": 17,
    "minor": 10,
    "patch": 0
  },
  "current_database": "pydbadmin_dev",
  "current_user": "postgres"
}
{
  "engine": "postgresql",
  "version": {
    "major": 17,
    "minor": 10,
    "patch": 0
  },
  "current_database": "pydbadmin_dev",
  "current_user": "postgres"
}
▶ pydbadminkit --connection local-native --config C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml --output yaml database list
- name: postgres
  owner: postgres
  encoding: UTF8
  collation: French_France.1252
  allow_connections: true
  connection_limit: -1
  size_bytes: 8033971
- name: pydbadmin_dev
  owner: postgres
  encoding: UTF8
  collation: French_France.1252
  allow_connections: true
  connection_limit: -1
  size_bytes: 8050355


CompletedProcess(args=['c:\\Users\\awounfouet\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pydbadminkit', '--connection', 'local-native', '--config', 'C:\\Users\\awounfouet\\Projects\\packages\\pydbadminkit\\config.toml', '--output', 'yaml', 'database', 'list'], returncode=0, stdout='- name: postgres\n  owner: postgres\n  encoding: UTF8\n  collation: French_France.1252\n  allow_connections: true\n  connection_limit: -1\n  size_bytes: 8033971\n- name: pydbadmin_dev\n  owner: postgres\n  encoding: UTF8\n  collation: French_France.1252\n  allow_connections: true\n  connection_limit: -1\n  size_bytes: 8050355\n', stderr='')

## 6. Nettoyage optionnel


In [19]:
run_mutation(
    [
        "--yes",
        "access",
        "revoke",
        "--role",
        "demo_user",
        "--object",
        "public.customers",
        "--access",
        "SELECT",
    ]
)
run_mutation(["--yes", "role", "membership-remove", "demo_reader", "demo_user"])
run_mutation(["--yes", "role", "drop", "demo_user"])
run_mutation(["--yes", "role", "drop", "demo_reader"])

[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)
[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)
[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)
[SKIP] Mutation réelle désactivée (RUN_MUTATIONS=False)


In [20]:
if not RUN_MUTATIONS:
    print("[SKIP] Nettoyage des objets SQL de démonstration")
else:
    import psycopg

    password = os.environ[PASSWORD_ENV]
    with (
        psycopg.connect(
            host=resolved_config.host,
            port=resolved_config.port,
            dbname=resolved_config.database,
            user=resolved_config.username,
            password=password,
        ) as connection,
        connection.cursor() as cursor,
    ):
        cursor.execute("DROP VIEW IF EXISTS public.active_customers")
        cursor.execute("DROP TABLE IF EXISTS public.customers")
    print("Objets SQL de démonstration supprimés.")

[SKIP] Nettoyage des objets SQL de démonstration


## Commandes couvertes

| Commande | Rôle |
|---|---|
| `connection test` | tester le profil sélectionné |
| `capability list` | lister les capabilities |
| `server info` | inspecter le serveur |
| `database/schema/table/view/index` | Object Explorer |
| `role/access/effective-access/ownership` | Security read-only |
| `role create/alter/drop` | mutations de rôles |
| `role membership-add/membership-remove` | memberships |
| `access grant/revoke` | ACL relationnelles |

Les mutations réelles restent opt-in via `RUN_MUTATIONS`.
